# 04 - Tools

Covers: defining a tool with `@tool`, inspecting its schema, binding it to a model, and executing the tool call the model requests.

In [ ]:
from llm_engineering import get_llm

llm = get_llm()

## Defining a tool

The name, docstring and type hints become the schema the model sees, so write them carefully.

In [ ]:
from langchain_core.tools import tool


@tool
def calculate_discount(price: float, discount_percentage: float) -> float:
    """
    Calculates the final price after applying a discount.

    IMPORTANT:
    - discount_percentage MUST be between 0 and 100
    - Values outside this range are INVALID
    - Do NOT reinterpret invalid values (e.g., 120% should NOT become -20%)
    - If invalid input is given, this tool should not be used

    Args:
        price (float): The original price of the item.
        discount_percentage (float): Discount percentage (0-100 only).

    Returns:
        float: The final price after the discount is applied.
    """
    if not 0 <= discount_percentage <= 100:
        raise ValueError("Discount percentage must be between 0 and 100")

    return price - price * (discount_percentage / 100)

In [ ]:
print(calculate_discount.name)
print(calculate_discount.description)
print(calculate_discount.args)

try:
    print(calculate_discount.invoke({"price": 100, "discount_percentage": 15}))
except ValueError as e:
    print("Error:", e)

## Binding the tool to a model

The model does not run the tool; it only returns a `tool_calls` request.

In [ ]:
llm_with_tools = llm.bind_tools([calculate_discount])

greeting = llm_with_tools.invoke("Hello world!")
print("Content:", greeting.content)
print("Tool calls:", greeting.tool_calls)  # empty: no tool needed

In [ ]:
query = """
What is the price of an item that costs $100 after a 20% discount?
If invalid, do NOT call the tool and explain why.
"""

result = llm_with_tools.invoke(query)

print("Content:", result.content)
print("Tool calls:", result.tool_calls)

## Executing the tool call

In [ ]:
if result.tool_calls:
    args = result.tool_calls[0]["args"]

    try:
        print("Final price:", calculate_discount.invoke(args))
    except ValueError as e:
        print("Tool error:", e)
else:
    print("Content:", result.content)